# 小松のおろし風167日を、地点の近さを加味して判定し直す (v1)

VS Code で上から順に実行してください。**このリポジトリのフォルダの中だけで
完結します。**

## 何をするか

モデルは日本全体の天気図を見て判断するので、小松に関係のない気圧配置を1位に
出すことがあります。一方 Grad-CAM は**モデルがどこを見てそのラベルを出したか**
を示します。そこで、

    点数 = 確信度 × (小松への近さで重み付けした熱の割合 ** ATTENTION_WEIGHT)

で並べ替え、**元の1位と、付け替えたあとの1位を突き合わせます。**

## 読むときの注意

**冬型や前線のように広域の配置で決まるラベルは、構造的に不利になります。**
根拠が日本全体に広がるので、小松の近くに集まる熱の割合が小さくなるためです。
これは「精度を上げる手法」ではなく、**「小松の近くにある系で説明したい」
という見方に切り替える道具**だと考えてください。

処理には**20〜40分ほどかかります**(1枚あたり数秒 × 167日)。

In [ ]:
# セットアップ(最初に1回だけ)
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

# **どのPythonで動いているかを必ず出す。**VS Code のカーネルは、ターミナルで
# 有効化した仮想環境とは別のものが選ばれていることがある
print(f"リポジトリ: {ROOT}")
print(f"Python    : {sys.executable}")

%matplotlib inline

import pandas as pd

from src.quicklook import rerank_by_site, rerank_dates
from src.sites import get_site

_site = get_site("komatsu")
print(f"\n地点: {_site.name}  中心 ({_site.x}, {_site.y})  半径 {_site.radius}")
print(f"  {_site.note}")

## 設定

ここだけ書き換えれば、他はそのままで動きます。

In [ ]:
SITE = "komatsu"
DATES_CSV = ROOT / "data" / "oroshi_komatsu_dates.csv"
DATE_COLUMN = "発生日"
HOUR = 0                 # 0 か 12

# 0 = 元の順位のまま / 1 = 近さを全面的に効かせる
ATTENTION_WEIGHT = 1.0

# "proximity" = 距離で滑らかに重み付け(既定) / "circle" = 円の中だけ
MODE = "proximity"

# 重みが約37%まで落ちる距離。None なら data/sites.csv の半径(0.12)
SCALE = None

OUT = ROOT / "reports" / "komatsu_rerank.csv"

print(f"日付: {DATES_CSV}")
print(f"出力: {OUT}")

## 実行

**ここが時間のかかるセルです。**進み具合と残り時間が出ます。

In [ ]:
result = rerank_dates(
    DATES_CSV, site=SITE, hour=HOUR, date_column=DATE_COLUMN,
    mode=MODE, scale=SCALE, attention_weight=ATTENTION_WEIGHT,
    out=OUT, progress_every=10,
)

## 結果の表

`reports/komatsu_rerank.csv` にも同じものが入っています(Excelでそのまま開けます)。

In [ ]:
# 1位が入れ替わった日だけを見る
_moved = result[result["入れ替わった"]]
print(f"入れ替わった日: {len(_moved)}日 / {len(result)}日")

_moved[["発生日", "元の1位", "元の確信度", "元の1位の集中度",
        "新しい1位", "新しい1位の確信度", "新しい1位の集中度"]]

## 図: 1位のラベルの内訳(元 / 付け替え後)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from src.jp_font import missing_font_hint, register_matplotlib_cjk

if not register_matplotlib_cjk():
    print("警告: 日本語フォントが見つかりません。" + missing_font_hint())

_before = result["元の1位"].value_counts()
_after = result["新しい1位"].value_counts()
_labels = sorted(set(_before.index) | set(_after.index),
                 key=lambda l: int(_before.get(l, 0)))
_b = [int(_before.get(l, 0)) for l in _labels]
_a = [int(_after.get(l, 0)) for l in _labels]

_y = np.arange(len(_labels))
_height = 0.38
fig, ax = plt.subplots(figsize=(9, max(4, len(_labels) * 0.55)))
_bars_b = ax.barh(_y + _height / 2, _b, _height, label="元の1位", color="#4C72B0")
_bars_a = ax.barh(_y - _height / 2, _a, _height, label="近さを加味した1位",
                  color="#DD8452")

# 数値を棒の脇に直接置く(色だけに頼らせない)
for _bars in (_bars_b, _bars_a):
    for _bar in _bars:
        _w = _bar.get_width()
        if _w:
            ax.text(_w + max(_a + _b) * 0.01, _bar.get_y() + _bar.get_height() / 2,
                    str(int(_w)), va="center", fontsize=9, color="#333333")

ax.set_yticks(_y)
ax.set_yticklabels(_labels, fontsize=10)
ax.set_xlabel("日数")
ax.set_title(f"おろし風{len(result)}日の1位ラベル(小松への近さを加味する前と後)",
             pad=14)
ax.legend(loc="lower right", frameon=False)
ax.grid(axis="x", alpha=0.25)
ax.set_axisbelow(True)
for _side in ("top", "right", "left"):
    ax.spines[_side].set_visible(False)
fig.tight_layout()
fig.savefig(ROOT / "reports" / "komatsu_rerank.png", dpi=150)
plt.show()
print("図: reports/komatsu_rerank.png")

## 入れ替わった日を1枚ずつ確かめる

**数字だけで判断しないでください。**ヒートマップに円を重ねた絵で、
「なぜ下がったか」が見えます。

In [ ]:
LIMIT = 3   # 一度に見る枚数

for _date in _moved["発生日"].head(LIMIT):
    rerank_by_site(_date, hour=HOUR, site=SITE, mode=MODE, scale=SCALE,
                   attention_weight=ATTENTION_WEIGHT, top_k=3)
    print("=" * 70)